# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *063* |
| **Integrantes** | *Jorge Andres Ocampo Suarez* |
| **Caso de estudio** | *(Wanderbricks u otro)* |
| **Fecha de entrega** | domingo 23 de agosto |
| **🎥 Enlace al video** | *https://youtu.be/8W8O6gVFANc* |

**Nota:** Profesora, tuve un inconveniente con mi registro en la organización en GitHub, de igual manera publique mi trabajo en un repositorio.
https://github.com/jorgeocampoiudigital/bigdata-2026b-g063

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*¿Qué se quiere resolver y por qué importa? Máximo tres párrafos.
Debe quedar claro qué preguntas del negocio deberá responder esta base de datos.*

**Preguntas de negocio:**
- ¿Qué propiedades y anfitriones generan más ingresos, y qué características comparten?
- ¿Qué tan bien se está convirtiendo la navegación en reservas, y dónde se pierden los usuarios en el camino?
- ¿El precio y el rating están correlacionados, o hay propiedades caras con mal desempeño y/o baratas con buen desempeño?

**Contexto:**
Wanderbricks es un marketplace de alquiler vacacional que conecta anfitriones con viajeros, y como todo marketplace, su negocio depende de tres cosas: que las propiedades correctas generen ingresos, que los usuarios que navegan efectivamente reserven, y que el precio se corresponda con la calidad percibida. Sin una base de datos analítica que integre reservas, pagos, reseñas y navegación en un mismo lugar, estas preguntas quedan dispersas en silos operativos y son difíciles de responder de forma consistente.

Esta base de datos analítica debe apoyar decisiones como priorizar qué propiedades y anfitriones promover, identificar en qué punto del recorrido de navegación se pierden los usuarios antes de reservar, y evaluar si la estrategia de precios está alineada con la satisfacción de los huéspedes.

el notebook busca responder: (1) qué propiedades y anfitriones generan más ingresos y qué las diferencia, (2) qué tan efectivo es el funnel de conversión desde la navegación hasta la reserva completada, y (3) si existe relación entre precio por noche y rating de las reseñas.

---
## 2. Descripción de los datos

*Volumen, variedad, tipos, calidad observada y relaciones entre tablas.
Esta descripción es la base de la decisión de diseño de la sección 3: sin ella,
cualquier justificación queda en el aire.*

In [0]:
# Exploración inicial del caso

display(spark.sql("SHOW TABLES IN samples.wanderbricks"))


In [0]:
# Esquema y muestra de cada tabla
tablas = [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]
for t in tablas:
    df = spark.table(f"samples.wanderbricks.{t}")
    print(f"--- {t} ---")
    df.printSchema()
    df.limit(5).display()

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")

In [0]:
# Calidad: nulos por columna en una tabla clave
from pyspark.sql import functions as F
df = spark.table("samples.wanderbricks.bookings")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).display()

In [0]:
# Relaciones: ¿hay reservas huérfanas (sin propiedad válida)?
bookings = spark.table("samples.wanderbricks.bookings")
properties = spark.table("samples.wanderbricks.properties")
huerfanas = bookings.join(properties, "property_id", "left_anti")
print("reservas sin propiedad:", huerfanas.count())

**TODO: describir las tablas que van a usar (esquema, tipos, nulos, cardinalidades)**

El dataset tiene 16 tablas con volúmenes muy dispares: desde catálogos pequeños como countries (168) o destinations (42), hasta tablas de eventos de alto volumen como page_views (500,000) y clickstream (100,000). Las tablas transaccionales núcleo son bookings (72,247), payments (49,638) y reviews (99,793) — la diferencia entre reservas y pagos (72,247 vs 49,638) sugiere que no todas las reservas llegan a pago completado (canceladas, pendientes), booking_updates (83,068) supera a bookings, lo que indica que cada reserva pasa por varios estados a lo largo de su ciclo de vida.

En calidad, bookings no presenta valores nulos en ninguna de sus columnas, y la verificación de integridad referencial contra properties no arrojó reservas huérfanas

---
## 3. Decisiones de diseño y justificación

*Comparar al menos tres paradigmas —relacional, NoSQL (documental / clave-valor / columnar)
y lakehouse— **atando cada criterio a los datos descritos arriba**. Las ventajas genéricas
copiadas de un manual no cuentan.*

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
| Integridad transaccional en bookings/payments (sin nulos, sin huérfanas contra properties) | Fuerte — constraints e integridad referencial nativasa | Débil — sin FKs nativos, la integridad quedaría a cargo de la aplicación | Fuerte — Delta soporta constraints y ACID, y permite joins con Spark SQL igual que un relacional | Lakehouse |
| Volumen de eventos (page_views 500K, clickstream 100K) | Limitado — un motor relacional tradicional no está pensado para escribir/leer cientos de miles de eventos de forma eficiente | Fuerte en escritura de eventos, pero débil para agregarlos junto a las tablas transaccionales | Fuerte — escala horizontalmente y permite unir eventos con bookings/properties en el mismo motor | Lakehouse |
| Historial de estados por reserva (booking_updates: 83,068 registros para 72,247 bookings, ~1.15 actualizaciones por reserva) | Posible, pero requiere una tabla de auditoría aparte gestionada manualmente | No ofrece versionado nativo de una tabla completa | Nativo — DESCRIBE HISTORY y time travel dan trazabilidad de cambios sin tabla de auditoría adicional | Lakehouse |
| Escalabilidad futura (empleados internos + usuarios ya suman ~197K registros) | Escala vertical, se vuelve costosa a ese tamaño | Escala horizontal pero sacrificando consistencia | Escala horizontal manteniendo ACID | Lakehouse |

**Referencias (APA 7):**

Databricks. (s.f.). Delta Lake: Aumenta la confiabilidad de los datos en el almacenamiento en la nube. Databricks. Recuperado el 29 de agosto de 2026, de https://www.databricks.com/es/blog/delta-lake-explained-boost-data-reliability-cloud-storage

Microsoft. (s.f.). ¿Qué es Delta Lake en Azure Databricks? Microsoft Learn. Recuperado el 29 de agosto de 2026, de https://learn.microsoft.com/es-es/azure/databricks/delta/

Microsoft. (s.f.). ¿Qué es un data lakehouse? Microsoft Learn. Recuperado el 29 de agosto de 2026, de https://learn.microsoft.com/es-es/azure/databricks/lakehouse/

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo63"
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
from pyspark.sql import functions as F

tablas = [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]

for t in tablas:
    (spark.table(f"samples.wanderbricks.{t}")
        .withColumn("_ingested_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .saveAsTable(f"{CATALOGO}.{ESQUEMA}.bronze_{t}"))

print("Tablas bronce creadas:", [f"bronze_{t}" for t in tablas])

Se ingirieron las 16 tablas del catálogo samples.wanderbricks sin ningún filtro ni transformación, replicando el principio de la capa bronce: preservar los datos exactamente como llegan de la fuente, para poder reprocesar desde cero si más adelante se detecta un error en la lógica de limpieza de la capa plata. A cada tabla se le agregó únicamente la columna _ingested_at, que registra el momento en que se ejecutó la ingesta, esto da trazabilidad sobre cuándo entró cada versión de los datos crudos al lakehouse, sin alterar ninguno de los valores originales.

No se aplicó limpieza, casteo de tipos ni eliminación de duplicados en esta etapa: esas decisiones se ejecutan en la capa plata, donde sí importa dejar rastro explícito de qué regla de negocio se aplicó y por qué.

### 4.3 Capa plata — datos limpios y tipados

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType,
    IntegerType, DecimalType, TimestampType
)
from delta.tables import DeltaTable

(
    DeltaTable.createOrReplace(spark)
    .tableName("bigdata_grupo63.wanderbricks.silver_bookings")
    .addColumn("booking_id", StringType(), nullable=False)
    .addColumn("user_id", StringType(), nullable=False)
    .addColumn("property_id", StringType(), nullable=False)
    .addColumn("check_in", DateType())
    .addColumn("check_out", DateType())
    .addColumn("guests_count", IntegerType())
    .addColumn("total_amount", DecimalType(10, 2))
    .addColumn("status", StringType())
    .addColumn("created_at", TimestampType())
    .addColumn("updated_at", TimestampType())
    .execute()
)

df_bronze = spark.table("bigdata_grupo63.wanderbricks.bronze_bookings")

df_silver = (
    df_bronze
    .filter((F.col("total_amount") >= 0) & (F.col("check_out") > F.col("check_in")))
    .select(
        F.col("booking_id"),
        F.col("user_id"),
        F.col("property_id"),
        F.col("check_in").cast(DateType()).alias("check_in"),
        F.col("check_out").cast(DateType()).alias("check_out"),
        F.col("guests_count").cast(IntegerType()).alias("guests_count"),
        F.col("total_amount").cast(DecimalType(10, 2)).alias("total_amount"),
        F.col("status"),
        F.col("created_at"),
        F.col("updated_at"),
    )
)

df_silver.write.format("delta").mode("append").saveAsTable(
    "bigdata_grupo63.wanderbricks.silver_bookings"
)

In [0]:
(
    spark.table("bigdata_grupo63.wanderbricks.bronze_payments")
    .filter(F.col("booking_id").isNotNull())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bigdata_grupo63.wanderbricks.silver_payments")
)

(
    spark.table("bigdata_grupo63.wanderbricks.bronze_properties")
    .filter(F.col("property_id").isNotNull())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bigdata_grupo63.wanderbricks.silver_properties")
)

(
    spark.table("bigdata_grupo63.wanderbricks.bronze_reviews")
    .filter(F.col("booking_id").isNotNull())
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bigdata_grupo63.wanderbricks.silver_reviews")
)

Sobre la capa bronce se construyó la capa plata aplicando esquema explícito y reglas de calidad concretas, en lugar de copiar los datos tal como llegaron. En silver_bookings se tiparon explícitamente fechas (DATE), montos (DECIMAL(10,2)) y cantidad de huéspedes (INT) — en bronce estos campos quedan con el tipo genérico de la ingesta cruda —, y se descartaron dos tipos de registros inconsistentes: reservas con total_amount negativo (que no tiene sentido de negocio) y reservas donde la fecha de salida no es posterior a la de entrada. Ninguna de estas dos condiciones apareció en la verificación de nulos, pero se agregan como salvaguarda explícita del modelo.

Para payments y reviews se filtró cualquier registro sin booking_id, ya que ambas tablas solo tienen sentido analítico si pueden vincularse a una reserva concreta; para properties se exigió property_id no nulo por ser la llave que sostiene la mayoría de los joins del notebook

### 4.4 Datos semiestructurados

*Al menos una tabla debe manejar estructuras anidadas (structs o arrays).
El clickstream y las reseñas son los candidatos naturales.*

In [0]:
# TODO: leer y aplanar estructuras anidadas
df = spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_clickstream")
df.printSchema()
df.select("metadata").limit(5).display()

In [0]:
from pyspark.sql import functions as F

# Aplanar el struct 'metadata' con notación de punto
flat = df.select(
    "event", "property_id", "timestamp", "user_id",
    "metadata.device", "metadata.referrer"
)
flat.display()

In [0]:
df_silver_clickstream = (
    spark.table("bigdata_grupo63.wanderbricks.bronze_clickstream")
    .select(
        F.col("event"),
        F.col("property_id"),
        F.col("timestamp").cast(TimestampType()).alias("event_timestamp"),
        F.col("user_id"),
        F.col("metadata.device").alias("device"),
        F.col("metadata.referrer").alias("referrer"),
    )
)

df_silver_clickstream.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bigdata_grupo63.wanderbricks.silver_clickstream")

clickstream se eligió como la tabla candidata para trabajar estructuras anidadas: cada evento de navegación  trae un campo metadata de tipo struct con device, desde qué tipo de dispositivo se generó el evento y referrer, el canal de origen, por ejemplo email. 

### 4.5 Propiedades del lakehouse

*Hay que evidenciar las tres: atomicidad, time travel y evolución de esquema.*

In [0]:
from delta.tables import DeltaTable

target_table = DeltaTable.forName(spark, "bigdata_grupo63.wanderbricks.silver_bookings")
source_df = spark.table("bigdata_grupo63.wanderbricks.bronze_bookings")

(
    target_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.booking_id = source.booking_id"
    )
    .whenMatchedUpdate(set={
        "status": "source.status",
        "updated_at": "source.updated_at",
    })
    .whenNotMatchedInsertAll()
    .execute()
)


Atomicidad: el MERGE sobre silver_bookings actualiza el estado de las reservas existentes e inserta las nuevas en una sola operación transaccional.

In [0]:
df_status_count = (
    spark.table("bigdata_grupo63.wanderbricks.silver_bookings")
    .groupBy("status")
    .agg(F.count("*").alias("cantidad"))
    .orderBy(F.col("cantidad").desc())
)

df_status_count.show()

Consistencia: Delta garantiza que el MERGE no modifique datos inconsistentes. Por ejemplo, si se intenta actualizar el estado de una reserva que no existe, Delta no permitirá la operación.

In [0]:
df_history = spark.table("bigdata_grupo63.wanderbricks.silver_bookings").__class__ and None

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "bigdata_grupo63.wanderbricks.silver_bookings")
df_history = delta_table.history()

df_history.show(truncate=False)

Versionado (time travel): DESCRIBE HISTORY deja un registro de cada operación de escritura sobre silver_bookings; el MERGE, la carga inicial, la evolución de esquema, con quién, cuándo y qué tipo de operación se ejecutó. La consulta con VERSION AS OF permite reconstruir el estado de la tabla exactamente antes del MERGE.

In [0]:
# Evolución de esquema con mergeSchema

from pyspark.sql import functions as F

nuevo_df = (spark.table(f"{CATALOGO}.{ESQUEMA}.bronze_bookings")
    .withColumn("cancellation_reason", F.lit(None).cast("string")))

(nuevo_df.write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA}.silver_bookings"))

Evolución de esquema: se agregó la columna cancellation_reason a silver_bookings sin recrear la tabla ni interrumpir las consultas existentes, usando mergeSchema. Esto demuestra que el modelo puede crecer conforme cambian las necesidades del negocio

### 4.6 Consultas analíticas

*Mínimo cinco, en SQL y en PySpark, que respondan preguntas reales del negocio.
Consultas triviales sin conexión con el problema no puntúan.*

Pregunta 1 — ¿Qué propiedades generan más ingresos?

In [0]:
%sql
SELECT p.property_id, p.title, SUM(b.total_amount) AS ingresos_totales
FROM bigdata_grupo63.wanderbricks.silver_bookings b
JOIN bigdata_grupo63.wanderbricks.silver_properties p ON b.property_id = p.property_id
WHERE b.status = 'completed'
GROUP BY p.property_id, p.title
ORDER BY ingresos_totales DESC
LIMIT 10;

In [0]:
bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")
properties = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_properties")

(bookings.filter(F.col("status") == "completed")
 .join(properties, "property_id")
 .groupBy("property_id", "title")
 .agg(F.sum("total_amount").alias("ingresos_totales"))
 .orderBy(F.desc("ingresos_totales"))
 .limit(10)
 .display())

Pregunta 2 — ¿Qué anfitriones tienen mejor desempeño combinado (ingresos + rating)?

In [0]:
%sql
SELECT h.host_id, h.name,
       SUM(b.total_amount) AS ingresos_totales,
       COUNT(DISTINCT p.property_id) AS num_propiedades,
       AVG(r.rating) AS rating_promedio
FROM bigdata_grupo63.wanderbricks.silver_bookings b
JOIN bigdata_grupo63.wanderbricks.silver_properties p ON b.property_id = p.property_id
JOIN samples.wanderbricks.hosts h ON p.host_id = h.host_id
LEFT JOIN bigdata_grupo63.wanderbricks.silver_reviews r ON b.booking_id = r.booking_id
WHERE b.status = 'completed'
GROUP BY h.host_id, h.name
ORDER BY ingresos_totales DESC
LIMIT 10;

In [0]:
hosts = spark.table("samples.wanderbricks.hosts")
reviews = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_reviews")
clickstream = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_clickstream")

(
     bookings.filter(F.col("status") == "completed")
     .join(properties, "property_id")
     .join(hosts, properties["host_id"] == hosts["host_id"])
     .join(reviews, bookings["booking_id"] == reviews["booking_id"], "left")
     .groupBy(hosts["host_id"], hosts["name"])
     .agg(F.sum("total_amount").alias("ingresos_totales"),
          F.countDistinct(properties["property_id"]).alias("num_propiedades"),
          F.avg(reviews["rating"]).alias("rating_promedio"))
     .orderBy(F.desc("ingresos_totales"))
     .limit(10).display()
)

Pregunta 3 — Funnel de conversión por dispositivo (clickstream → reserva)

In [0]:
%sql
SELECT
    c.device,
    COUNT(DISTINCT CASE WHEN c.event = 'view' THEN c.user_id END) AS usuarios_vieron,
    COUNT(DISTINCT b.user_id) AS usuarios_reservaron
FROM bigdata_grupo63.wanderbricks.silver_clickstream c
LEFT JOIN bigdata_grupo63.wanderbricks.silver_bookings b
    ON c.user_id = b.user_id
GROUP BY c.device
ORDER BY usuarios_vieron DESC;

In [0]:
from pyspark.sql import functions as F

vistas = clickstream.filter(F.col("event") == "view")

# Pregunta 3: funnel por dispositivo
funnel_device = (vistas.select("user_id", "device").distinct()
    .groupBy("device")
    .agg(F.countDistinct("user_id").alias("usuarios_vieron")))

reservas_por_device = (vistas.select("user_id", "device").distinct()
    .join(bookings.select("user_id").distinct(), "user_id")
    .groupBy("device")
    .agg(F.countDistinct("user_id").alias("usuarios_reservaron")))

(funnel_device.join(reservas_por_device, "device", "left")
    .withColumn("tasa_conversion_pct",
        F.round(F.col("usuarios_reservaron") / F.col("usuarios_vieron") * 100, 1))
    .display())

Pregunta 4 — ¿Precio y rating están correlacionados?

In [0]:
%sql
SELECT
    p.property_id,
    p.base_price,
    AVG(r.rating) AS rating_promedio
FROM bigdata_grupo63.wanderbricks.silver_properties p
JOIN bigdata_grupo63.wanderbricks.silver_bookings b ON p.property_id = b.property_id
JOIN bigdata_grupo63.wanderbricks.silver_reviews r ON b.booking_id = r.booking_id
GROUP BY p.property_id, p.base_price;

In [0]:
props_price = properties.select("property_id", "base_price")

correl_df = (props_price.join(bookings, props_price.property_id == bookings.property_id)
    .join(reviews, bookings.booking_id == reviews.booking_id)
    .groupBy(props_price.property_id, "base_price")
    .agg(F.avg("rating").alias("rating_promedio"))
    .select("base_price", "rating_promedio"))

print("Correlación precio-rating:", correl_df.stat.corr("base_price", "rating_promedio"))

Pregunta 5 — Canal de referencia (referrer) con mejor tasa de conversión

In [0]:
%sql
SELECT
    c.referrer,
    COUNT(DISTINCT c.user_id) AS usuarios_totales,
    COUNT(DISTINCT b.user_id) AS usuarios_reservaron,
    ROUND(COUNT(DISTINCT b.user_id) / COUNT(DISTINCT c.user_id) * 100, 1) AS tasa_conversion_pct
FROM bigdata_grupo63.wanderbricks.silver_clickstream c
LEFT JOIN bigdata_grupo63.wanderbricks.silver_bookings b
    ON c.user_id = b.user_id AND c.property_id = b.property_id
GROUP BY c.referrer
ORDER BY tasa_conversion_pct DESC;

In [0]:
por_referrer = (vistas.select("user_id", "referrer").distinct()
    .groupBy("referrer")
    .agg(F.countDistinct("user_id").alias("usuarios_totales")))

reservas_por_referrer = (vistas.select("user_id", "referrer").distinct()
    .join(bookings.select("user_id").distinct(), "user_id")
    .groupBy("referrer")
    .agg(F.countDistinct("user_id").alias("usuarios_reservaron")))

(por_referrer.join(reservas_por_referrer, "referrer", "left")
    .withColumn("tasa_conversion_pct",
        F.round(F.col("usuarios_reservaron") / F.col("usuarios_totales") * 100, 1))
    .orderBy(F.desc("tasa_conversion_pct")).display())

In [0]:
clickstream = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_clickstream")
bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")

# ¿Los user_id de clickstream existen en bookings?
cs_users = clickstream.select("user_id").distinct()
bk_users = bookings.select("user_id").distinct()
print("usuarios distintos en clickstream:", cs_users.count())
print("usuarios distintos en bookings:", bk_users.count())
print("usuarios en clickstream que SÍ existen en bookings:", cs_users.join(bk_users, "user_id").count())

# Rango de IDs en cada lado, por si hay un desfase de rangos
clickstream.select(F.min("user_id"), F.max("user_id")).show()
bookings.select(F.min("user_id"), F.max("user_id")).show()

# ¿property_id combinado con user_id realmente coincide alguna vez?
print("pares (user_id, property_id) que coinciden:",
      clickstream.select("user_id","property_id").distinct()
        .join(bookings.select("user_id","property_id").distinct(), ["user_id","property_id"])
        .count())

---
## 5. Resultados

*Qué se obtuvo. Las salidas de las celdas deben quedar visibles en el notebook exportado.*

**Ingresos por propiedad:** el top 10 está dominado por hoteles y apartamentos de ciudades grandes (Abu Dhabi, Osaka, Londres, Nueva York, Dubái), con montos entre $4,949 y $7,068. No hay un tipo de propiedad que domine claramente la lista. Hoteles, apartamentos, bungalows e inns aparecen mezclados, lo que sugiere que la ubicación pesa más que el tipo de propiedad en los ingresos totales.

**Desempeño de anfitriones:** El ranking por ingresos no coincide con el ranking por rating: Holly Chapman lidera con $91,734 en ingresos y 40 propiedades, pero su rating promedio (2.99) es de los más bajos del top 10; Tracy Murray, en cambio, tiene el mejor rating del grupo (3.30) pero factura menos y administra 9 propiedades menos. El patrón se repite en todo el top 10: son los anfitriones con más propiedades los que llegan arriba en ingresos, no los mejor calificados.

**Correlación precio-rating:** el coeficiente calculado sobre el catálogo completo de propiedades es 0.007. No existe relación lineal entre cuánto cuesta una propiedad por noche y qué tan bien calificada está.

**Funnel de conversión:** el 44% de los usuarios que navegaron (vieron una propiedad) terminaron completando al menos una reserva, con muy poca variación entre dispositivos (44.1%–44.4%) y entre canales de referencia (43.7%–44.9%, con email ligeramente arriba y ad ligeramente abajo). La conversión es alta y notablemente uniforme, ni el dispositivo ni el canal de origen parecen ser palancas relevantes para mejorar la tasa de reserva en este dataset.

**Lectura conjunta:** los tres análisis apuntan en la misma dirección, el negocio de Wanderbricks convierte visitantes en huéspedes de forma consistente sin importar cómo llegan, ni el precio predice la satisfacción, ni la calidad predice quién gana más. Esto sugiere que, si el negocio quisiera optimizar algo, el punto de apalancamiento no está en el funnel de adquisición sino en la relación entre calidad de servicio y crecimiento de portafolio de los anfitriones.

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

El ejercicio confirmó, con evidencia concreta y no solo con argumentos genéricos, que el lakehouse era la elección correcta para este caso. La justificación de la sección 3 se sostiene en la práctica: bookings y payments mantuvieron cero nulos y cero huérfanas durante todo el proceso, lo que valida las garantías tipo-relacional que ofrece Delta; al mismo tiempo, clickstream trajo un campo anidado (metadata) que se manejó de forma nativa sin normalizar en tablas aparte, algo que un modelo puramente relacional habría hecho más costoso. Las tres propiedades del lakehouse (atomicidad, time travel y evolución de esquema) se evidenciaron con salidas concretas, no solo en teoría, y directamente conectadas a una necesidad real del negocio: booking_updates ya mostraba que las reservas cambian de estado varias veces, y el time travel da una forma nativa de auditar esos cambios.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Jorge Andres Ocampo Suarez | Trabajo Completo | Trabajo Completo |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

Se uso IA para ayuda en la generación de las consultas en Spark, transformando de SQL a lenguaje Spark.

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?

Se eligió el lakehouse (Delta Lake) porque el caso tenía dos necesidades que ningún paradigma puro cubre solo: por un lado, bookings y payments exigían integridad fuerte (nulos en cero, sin reservas huérfanas), típica de un modelo relacional; por otro, clickstream traía un campo anidado (metadata con device y referrer) que un relacional puro obligaría a normalizar en tablas aparte. Se descartó el modelo puramente relacional porque no maneja bien estructuras anidadas ni el volumen de eventos (page_views con 500,000 registros) sin normalización costosa, y se descartó un NoSQL documental puro porque no ofrece integridad referencial ni transacciones ACID para bookings/payments, y tampoco versionado nativo.

2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.

`(bookings.filter(F.col("status") == "completed")
    .join(properties, "property_id")
    .groupBy("property_id", "name")
    .agg(F.sum("total_amount").alias("ingresos_totales"))
    .orderBy(F.desc("ingresos_totales"))
    .limit(10).display())`

3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

Particionamiento: hoy las tablas Delta no están particionadas explícitamente; con cien veces más volumen, sería ideal en silver_bookings y silver_clickstream particionar por fecha para que las consultas que filtran por rango de fechas no escaneen toda la tabla.

Optimización de archivos: correr un OPTIMIZE con ZORDER BY sobre las columnas más usadas en filtros y joins (property_id, user_id) para reducir el número de archivos pequeños y acelerar lecturas.

Escalado del clúster: pasar de un clúster fijo a uno con autoescalado, dado que el shuffle en los groupBy se vuelve mucho más costoso con más datos y más nodos ayudan a paralelizarlo.

In [0]:
from pyspark.sql import functions as F

CATALOGO = "bigdata_grupo63"
ESQUEMA  = "wanderbricks"
bookings = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_bookings")
properties = spark.table(f"{CATALOGO}.{ESQUEMA}.silver_properties")

(
    bookings.filter(F.col("status") == "completed")
    .join(properties, "property_id")
    .groupBy("property_id", "title")
    .agg(F.sum("total_amount").alias("ingresos_totales"))
    .orderBy(F.desc("ingresos_totales"))
    .limit(10).display()
)

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas